# Karpenter Cluster Autoscaler

A practical refresher on **Karpenter** — the open-source, just-in-time Kubernetes node-provisioning autoscaler built by AWS. Karpenter watches for *unschedulable* pods and launches **right-sized nodes directly** (no fixed node groups), then continuously **consolidates** the fleet to cut cost. It is the modern replacement for the legacy Cluster Autoscaler on EKS and is especially valuable for bursty, heterogeneous ML/GPU workloads.

## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)

## Introduction

**Karpenter** is a Kubernetes controller that provisions compute *just in time*. Instead of scaling a fixed set of Auto Scaling Groups (ASGs) up and down like the classic Cluster Autoscaler, Karpenter looks at the **actual resource requests and scheduling constraints of pending pods** and launches the cheapest instance(s) that satisfy them — directly through the EC2 `CreateFleet` API. It then watches the running fleet and removes or replaces nodes that are empty, underutilized, drifted, or expired.

Karpenter reached **v1.0 (GA) in August 2024**; the stable API uses two CRDs — `NodePool` (`karpenter.sh/v1`) and, on AWS, `EC2NodeClass` (`karpenter.k8s.aws/v1`). These replaced the earlier `Provisioner`/`AWSNodeTemplate` (v1alpha5) and the v1beta1 `NodePool`/`EC2NodeClass`.

### What is it?

A *groupless*, application-driven autoscaler. You declare **constraints** (allowed instance families, architectures, zones, capacity types, limits, disruption rules) in a `NodePool`, and **node configuration** (AMI, subnets, security groups, IAM role, disks, user data) in an `EC2NodeClass`. Karpenter does the bin-packing and instance selection itself, so a single NodePool can flexibly span dozens of instance types and both Spot and On-Demand capacity.

### Why use it?

- **Right-sized, just-in-time nodes** — picks the cheapest instance shape that fits the pending pods instead of forcing them into pre-defined ASGs.
- **Fast scale-up** — bypasses ASGs and provisions nodes in seconds-to-low-minutes by calling EC2 Fleet directly.
- **Built-in consolidation** — continuously removes/replaces underutilized nodes to lower cost without you writing scale-down logic.
- **First-class Spot** — diversifies across many instance types using the *price-capacity-optimized* strategy and handles interruptions gracefully.
- **Less node-group sprawl** — one or two NodePools replace dozens of hand-tuned ASGs/managed node groups.

### When to use it?

- **EKS clusters with dynamic or bursty demand** — CI runners, batch/ML training, data pipelines, inference that scales with traffic.
- **Heterogeneous / GPU workloads** — let Karpenter choose among many GPU and CPU families per job rather than maintaining one node group per shape.
- **Cost-sensitive Spot-heavy fleets** — broad instance diversification maximizes Spot availability and savings.
- **Reach for something else** when you are not on a Karpenter-supported provider (today primarily AWS/EKS; community providers exist for Azure and others), when strict node-group pinning is a hard requirement, or for very small static clusters where any autoscaler is overkill.

## Key Features

### Core Capabilities of Karpenter

| Feature | Description | Benefit |
|---------|-------------|---------|
| Just-in-time provisioning | Reads pending-pod requests/constraints and launches matching nodes via EC2 Fleet | No pre-defined node groups; right-sized capacity in ~seconds |
| Groupless / flexible NodePools | One NodePool can span many instance families, sizes, zones, and capacity types | Better bin-packing and Spot diversification from a single config |
| Consolidation | Continuously removes empty nodes and replaces underutilized ones with cheaper options | Automatic, ongoing cost reduction |
| Disruption controls | Drift, Consolidation, Expiration, Interruption — with **disruption budgets** | Safe, rate-limited node churn that respects availability |
| Native Spot support | `price-capacity-optimized` allocation + SQS interruption handling | High Spot savings with graceful, pre-drained interruptions |
| Drift detection | Detects when a node no longer matches its NodePool/EC2NodeClass (e.g. new AMI) | Rolling, declarative node updates |
| `consolidateAfter` / `expireAfter` | Tunable timers for when to consolidate or recycle nodes | Trade churn against freshness/cost |
| Well-known requirement keys | Schedule on `karpenter.k8s.aws/instance-gpu-count`, `instance-category`, capacity-type, etc. | Precise, portable workload placement |
| `kubectl`-native CRDs | NodePool / EC2NodeClass managed via GitOps | Declarative, auditable infra-as-data |

## Architecture Overview

Karpenter runs as a Deployment (typically in `kube-system`) on the cluster. Its core loop is: **watch pending pods → compute the cheapest set of nodes that satisfies them → call EC2 `CreateFleet` → register the node**; and separately **watch the fleet → disrupt (consolidate/drift/expire) nodes safely**.

```
   kube-scheduler  --(pods stay Pending: no node fits)-->  Karpenter controller
                                                                 |
                                  +------------------------------+------------------------------+
                                  |  PROVISIONING                                               |
                                  |   1. batch pending pods, read requests + constraints        |
                                  |   2. solve: cheapest instance type(s) that fit (bin-pack)   |
                                  |   3. EC2 CreateFleet  --> launch instance(s)                |
                                  |   4. node joins; pods schedule                              |
                                  +-------------------------------------------------------------+
                                  |  DISRUPTION (continuous)                                    |
                                  |   - Consolidation: delete empty / replace underutilized     |
                                  |   - Drift: node config != NodePool/EC2NodeClass -> replace  |
                                  |   - Expiration: node older than expireAfter -> replace      |
                                  |   - Interruption: Spot/maintenance event (via SQS) -> drain |
                                  +-------------------------------------------------------------+
                                                                 |
                            NodePool (karpenter.sh/v1)  +  EC2NodeClass (karpenter.k8s.aws/v1)
                            constraints/limits/disruption    AMI, subnets, SGs, IAM role, disks
```

### Components

1. **Karpenter controller** — the Deployment that runs the provisioning and disruption reconcilers; needs an IAM role (IRSA or EKS Pod Identity) to call EC2.
2. **NodePool (`karpenter.sh/v1`)** — provider-agnostic constraints: allowed requirements (arch/zone/capacity-type/instance families), `limits`, `disruption` policy and budgets, `weight`, and the node template (labels/taints, `expireAfter`, `nodeClassRef`).
3. **EC2NodeClass (`karpenter.k8s.aws/v1`)** — AWS-specific node config: AMI family/selector, subnet & security-group selectors, IAM `role`, block-device mappings, user data, metadata options.
4. **NodeClaim** — an internal CRD Karpenter creates per launched node, tracking the lifecycle of a single provisioned instance.
5. **Interruption queue (SQS)** — receives Spot interruption / instance-rebalance / scheduled-maintenance events so Karpenter can cordon & drain proactively.

## Installation

### Prerequisites

- A running **EKS cluster** (Kubernetes 1.25+) and `kubectl` + `helm` v3 configured against it.
- An **OIDC provider** on the cluster (for IRSA) or **EKS Pod Identity** enabled.
- IAM resources: a **Karpenter controller role** (permissions for EC2 fleet/pricing/SSM) and a **node role/instance profile** (`KarpenterNodeRole-<cluster>`) that nodes assume.
- An **SQS interruption queue** plus EventBridge rules for Spot/maintenance events.
- **Tagged subnets and security groups** so the EC2NodeClass selectors can discover them — typically `karpenter.sh/discovery: <cluster-name>`.

> The full IAM/SQS/CloudFormation setup is automated by the official Getting Started guide and by Terraform/eksctl modules (shown in the *Production Deployment* section). The Helm chart installs the controller and CRDs.

### Installation Steps

Karpenter is a Kubernetes controller, **not** a Python package — there is nothing to `pip install`. It is installed with Helm from the public ECR OCI registry:

In [ ]:
# Karpenter is installed via Helm, not pip. These are the canonical commands
# (run in a shell against your EKS cluster). Shown here as reference, not executed.
INSTALL_COMMANDS = r"""
export CLUSTER_NAME=my-cluster
export KARPENTER_VERSION=1.1.1

# 1. Install the controller + CRDs from the public ECR OCI chart.
helm upgrade --install karpenter \
  oci://public.ecr.aws/karpenter/karpenter \
  --version "${KARPENTER_VERSION}" \
  --namespace kube-system \
  --set "settings.clusterName=${CLUSTER_NAME}" \
  --set "settings.interruptionQueue=${CLUSTER_NAME}" \
  --set controller.resources.requests.cpu=1 \
  --set controller.resources.requests.memory=1Gi \
  --wait

# 2. Confirm the controller is healthy and the CRDs are registered.
kubectl get deployment karpenter -n kube-system
kubectl api-resources | grep karpenter
#   nodepools        karpenter.sh
#   nodeclaims       karpenter.sh
#   ec2nodeclasses   karpenter.k8s.aws
"""
print(INSTALL_COMMANDS)


## Basic Usage

### Quick Start Example

Using Karpenter means applying two objects and then deploying normal workloads — Karpenter reacts to whatever Pending pods appear. A minimal, production-shaped pair looks like this:

- An **`EC2NodeClass`** that tells Karpenter *how* to build a node (AMI, where to put it, what role it gets).
- A **`NodePool`** that tells Karpenter *what kinds* of nodes are allowed and *when* to remove them.

The cell below defines that pair as YAML and runs a lightweight structural check of the required v1 fields, so you can see the exact shape you would `kubectl apply -f`.

In [ ]:
# Minimal Karpenter v1 NodePool + EC2NodeClass (AWS). Defined as YAML strings and
# sanity-checked for the required v1 fields. This is what you `kubectl apply -f`.

NODEPOOL_YAML = """
apiVersion: karpenter.sh/v1
kind: NodePool
metadata:
  name: default
spec:
  template:
    metadata:
      labels:
        team: ml-platform
    spec:
      requirements:
        - key: kubernetes.io/arch
          operator: In
          values: ["amd64"]
        - key: kubernetes.io/os
          operator: In
          values: ["linux"]
        - key: karpenter.sh/capacity-type
          operator: In
          values: ["spot", "on-demand"]
        - key: karpenter.k8s.aws/instance-category
          operator: In
          values: ["c", "m", "r"]
        - key: karpenter.k8s.aws/instance-generation
          operator: Gt
          values: ["4"]
      nodeClassRef:
        group: karpenter.k8s.aws
        kind: EC2NodeClass
        name: default
      expireAfter: 720h            # recycle nodes after 30 days
  limits:
    cpu: "1000"                    # hard ceiling: NodePool will not exceed this
    memory: 1000Gi
  disruption:
    consolidationPolicy: WhenEmptyOrUnderutilized
    consolidateAfter: 1m
    budgets:
      - nodes: "10%"               # disrupt at most 10% of this pool's nodes at once
"""

EC2NODECLASS_YAML = """
apiVersion: karpenter.k8s.aws/v1
kind: EC2NodeClass
metadata:
  name: default
spec:
  amiFamily: AL2023
  amiSelectorTerms:
    - alias: al2023@latest        # required in v1: pin or track the AMI explicitly
  role: "KarpenterNodeRole-my-cluster"
  subnetSelectorTerms:
    - tags:
        karpenter.sh/discovery: "my-cluster"
  securityGroupSelectorTerms:
    - tags:
        karpenter.sh/discovery: "my-cluster"
"""

REQUIRED = {
    "NodePool": ["apiVersion: karpenter.sh/v1", "nodeClassRef:", "disruption:", "consolidationPolicy:"],
    "EC2NodeClass": ["apiVersion: karpenter.k8s.aws/v1", "amiSelectorTerms:", "role:", "subnetSelectorTerms:"],
}
for kind, doc in (("NodePool", NODEPOOL_YAML), ("EC2NodeClass", EC2NODECLASS_YAML)):
    missing = [f for f in REQUIRED[kind] if f not in doc]
    status = "OK" if not missing else f"MISSING {missing}"
    print(f"{kind:13s} required v1 fields -> {status}")

print("\nApply with:  kubectl apply -f nodepool.yaml -f ec2nodeclass.yaml")


Now any unschedulable workload triggers provisioning. Scaling a Deployment beyond current capacity is enough:

```bash
# A pod with a 1-CPU request that the current nodes can't fit:
kubectl create deployment inflate --image=public.ecr.aws/eks-distro/kubernetes/pause:3.9
kubectl set resources deployment inflate --requests=cpu=1
kubectl scale deployment inflate --replicas=12

# Watch Karpenter pick instance types and launch nodes in real time:
kubectl logs -f -n kube-system -l app.kubernetes.io/name=karpenter
kubectl get nodeclaims        # the instances Karpenter created
kubectl get nodes -L karpenter.sh/capacity-type -L node.kubernetes.io/instance-type
```

## Advanced Features

### Disruption, Spot, weights, and GPU scheduling

#### Consolidation & disruption budgets

Karpenter's disruption engine reclaims waste continuously. `consolidationPolicy: WhenEmptyOrUnderutilized` lets it both delete empty nodes *and* replace/merge underutilized ones onto cheaper capacity; `WhenEmpty` is the conservative option that only removes fully empty nodes. **Disruption budgets** cap how much churn happens at once — by node count, percentage, or a cron `schedule`/`duration` window — so consolidation never threatens availability:

```yaml
disruption:
  consolidationPolicy: WhenEmptyOrUnderutilized
  consolidateAfter: 30s
  budgets:
    - nodes: "20%"                         # default: at most 20% disrupted
    - nodes: "0"                           # freeze disruption during business hours
      schedule: "0 9 * * mon-fri"
      duration: 8h
      reasons: ["Underutilized", "Drift"]  # but still allow Empty/Interruption
```

#### Spot with graceful interruption

Set `karpenter.sh/capacity-type` to include `spot`. Karpenter requests Spot with the **price-capacity-optimized** strategy and, via the SQS interruption queue, cordons and drains a node *before* the 2-minute Spot reclaim. Broadening the allowed instance types (don't over-constrain!) is what makes Spot reliable — more shapes means more spare-capacity pools to fall back on.

#### NodePool weights and layering

Multiple NodePools can coexist; `weight` orders them so Karpenter prefers, say, a Reserved/On-Demand baseline pool before falling through to a Spot pool. This models "reserved baseline + Spot burst" without separate ASGs.

#### GPU / accelerator scheduling (ML)

For ML training/inference, constrain a NodePool to GPU families and let pods request `nvidia.com/gpu`. Karpenter taints GPU nodes so only GPU pods land on them:

```yaml
# In NodePool spec.template.spec:
requirements:
  - key: karpenter.k8s.aws/instance-family
    operator: In
    values: ["g5", "g6", "p4d", "p5"]
taints:
  - key: nvidia.com/gpu
    effect: NoSchedule
# Pods then request `nvidia.com/gpu: 1` and tolerate the taint; the NVIDIA
# device plugin advertises the GPUs once the node is up.
```

In [ ]:
# What Karpenter does internally: choose the cheapest instance type that fits the
# *aggregate* of pending pod requests. This is a tiny illustration of that bin-pack
# decision (real Karpenter also weighs Spot pools, zones, and existing capacity).

# Pending pods (vCPU, GiB) waiting to schedule.
pending = [(2, 8), (2, 8), (4, 16), (1, 4), (1, 4)]
need_cpu = sum(c for c, _ in pending)
need_mem = sum(m for _, m in pending)

# A small candidate set with rough on-demand $/hr (us-east-1, illustrative only).
instances = [
    # name,        vCPU, GiB,  $/hr
    ("m6i.xlarge",    4,  16,  0.192),
    ("m6i.2xlarge",   8,  32,  0.384),
    ("m6i.4xlarge",  16,  64,  0.768),
    ("c6i.4xlarge",  16,  32,  0.680),
    ("r6i.2xlarge",   8,  64,  0.504),
]

# Cheapest SINGLE instance that fits everything (Karpenter may also split across
# several nodes; here we show the single-node choice for clarity).
fits = [(name, price) for name, cpu, mem, price in instances
        if cpu >= need_cpu and mem >= need_mem]
fits.sort(key=lambda x: x[1])

print(f"Pending pods need {need_cpu} vCPU and {need_mem} GiB in total.")
if fits:
    name, price = fits[0]
    print(f"Cheapest instance that fits: {name} at ${price}/hr "
          f"(${price * 24 * 30:.0f}/mo).")
    print("Rejected (too small or pricier):")
    for name2, price2 in [(n, p) for n, c, m, p in instances if (n, p) not in fits or (n, p) != fits[0]]:
        print(f"  - {name2} (${price2}/hr)")
else:
    print("No single instance fits; Karpenter would provision multiple nodes.")


## Use Cases

### Real-world Applications of Karpenter

#### Use Case 1: Bursty ML training / batch jobs

- **Context**: Data scientists submit training jobs at unpredictable times; each needs a specific GPU shape for a few hours.
- **Implementation**: A GPU NodePool spanning `g5`/`g6`/`p4d`/`p5` with `expireAfter` and aggressive consolidation; jobs request `nvidia.com/gpu`.
- **Results**: GPUs appear on demand, jobs run, and Karpenter tears the (expensive) nodes down the moment they go idle — no idle GPU bill.

#### Use Case 2: Cost-optimized Spot fleet for stateless services

- **Context**: A web/inference tier that tolerates interruption and wants maximum savings.
- **Implementation**: A broad Spot NodePool (many families/sizes), an On-Demand NodePool with higher `weight` as a small baseline, and pod anti-affinity for spread.
- **Results**: 60–90% compute savings versus On-Demand, with interruptions absorbed by graceful drain and rescheduling.

#### Use Case 3: Replacing dozens of managed node groups

- **Context**: A platform team maintains many ASGs/managed node groups per team, zone, and instance size — painful to tune.
- **Implementation**: Collapse them into a handful of NodePools with requirement ranges; let Karpenter pick shapes and bin-pack.
- **Results**: Far less config to maintain, better packing, and automatic right-sizing as workloads change.

## Best Practices

### Recommended Practices for Karpenter

1. **Keep requirements broad** — allow many instance families/sizes and both Spot + On-Demand. Over-constraining starves the solver, hurts bin-packing, and tanks Spot availability.
2. **Set resource *requests* accurately** — Karpenter provisions to pod **requests**. Wrong or missing requests cause over- or under-provisioning. Pair with the right-sizing recommendations from VPA/Goldilocks.
3. **Always set NodePool `limits`** — a `cpu`/`memory` ceiling is your guardrail against a runaway workload provisioning unbounded (and expensive) capacity.
4. **Use disruption budgets** — cap concurrent disruption (and optionally freeze it during peak hours) so consolidation never compromises availability.
5. **Embrace consolidation + `expireAfter`** — `WhenEmptyOrUnderutilized` plus a node TTL keeps the fleet cheap and AMIs fresh; use Drift for declarative, rolling node updates.
6. **Run the controller on stable capacity** — schedule Karpenter itself on a small On-Demand managed node group (or Fargate), never on nodes it manages, so it can't deprovision itself.
7. **Tag subnets/SGs consistently** — the `karpenter.sh/discovery` convention makes EC2NodeClass selectors predictable and GitOps-friendly.

## Common Pitfalls

### What to Avoid When Using Karpenter

1. **Over-constraining instance types** — pinning to a single family/size defeats Karpenter's bin-packing and makes Spot fragile. Let it choose from a wide pool.
2. **Karpenter scheduling on its own nodes** — if the controller runs on Karpenter-managed capacity, consolidation can evict it and stall scaling. Pin it to a managed node group/Fargate.
3. **Missing or sloppy pod requests** — no requests means Karpenter can't size nodes correctly; pods may stay Pending or land on oversized/undersized nodes.
4. **Forgetting NodePool `limits`** — without a ceiling, a misbehaving workload (or a `replicas` typo) can provision a huge, costly fleet.
5. **No interruption queue for Spot** — without the SQS queue + EventBridge rules, Spot reclaims become hard kills instead of graceful drains, causing avoidable disruption.
6. **Untagged or wrong-AZ subnets/SGs** — selectors silently match nothing and nodes never launch; check `kubectl describe nodeclaim` / EC2NodeClass status for discovery errors.
7. **Assuming it works off-EKS** — the mature provider is AWS. On other clouds you need a community provider; behavior and CRDs differ.

## Performance Optimization

### Optimizing Karpenter for Cost and Speed

#### Configuration Tuning

Key levers, roughly in order of impact:

- **Requirement breadth**: more allowed instance types ⇒ better packing, faster Spot fulfillment, and more consolidation options. This is the single biggest lever.
- **Capacity type mix**: prefer Spot with an On-Demand baseline (via NodePool `weight`) for the best price/availability balance.
- **`consolidationPolicy` + `consolidateAfter`**: shorter timers reclaim waste faster but increase churn; lengthen them for latency-sensitive or slow-starting pods.
- **Disruption budgets**: tune the percentage/schedule to balance savings against availability and pod-startup cost.
- **`expireAfter`**: recycle nodes to stay on fresh AMIs and avoid long-lived drift; longer TTLs reduce churn.
- **Startup taints / readiness**: use `startupTaints` so DaemonSets (CNI, device plugins) are ready before workloads schedule, avoiding crash-restart churn on new nodes.

The cell below sketches the kind of saving consolidation captures: replacing several lightly-loaded nodes with fewer right-sized ones.

In [ ]:
# Illustration of consolidation savings: a set of underutilized nodes whose pods
# can be repacked onto fewer/cheaper nodes. Karpenter does this continuously; here
# we just contrast "before" vs a simple repacked "after".

# Current fleet: (instance, vCPU, GiB, $/hr, used_cpu, used_mem)
current = [
    ("m6i.4xlarge", 16, 64, 0.768,  4, 18),
    ("m6i.4xlarge", 16, 64, 0.768,  3, 12),
    ("m6i.2xlarge",  8, 32, 0.384,  2,  9),
]
before_cost = sum(p for _, _, _, p, _, _ in current)
total_cpu = sum(c for _, _, _, _, c, _ in current)
total_mem = sum(m for _, _, _, _, _, m in current)

# After consolidation: everything (9 vCPU / 39 GiB used) repacks onto one node.
candidates = [
    ("m6i.2xlarge",  8, 32, 0.384),
    ("m6i.4xlarge", 16, 64, 0.768),
    ("r6i.2xlarge",  8, 64, 0.504),
]
fit = sorted([(n, p) for n, c, m, p in candidates if c >= total_cpu and m >= total_mem],
             key=lambda x: x[1])
after_node, after_cost = fit[0]

print(f"Before: {len(current)} nodes, ${before_cost:.3f}/hr "
      f"(used {total_cpu} vCPU / {total_mem} GiB across them)")
print(f"After:  1x {after_node}, ${after_cost:.3f}/hr")
saving = before_cost - after_cost
print(f"Saving: ${saving:.3f}/hr  (~${saving * 24 * 30:.0f}/mo, "
      f"{saving / before_cost:.0%} cheaper)")


## Production Deployment

### Deploying Karpenter in Production

You deploy the **controller** (Helm) plus the **IAM/SQS scaffolding**, then manage NodePools/EC2NodeClasses via GitOps. Most teams use a module rather than wiring IAM by hand.

#### Terraform (community `terraform-aws-eks` Karpenter submodule)

```hcl
module "karpenter" {
  source  = "terraform-aws-modules/eks/aws//modules/karpenter"
  version = "~> 20.0"

  cluster_name = module.eks.cluster_name

  enable_pod_identity             = true   # preferred over IRSA on new clusters
  create_pod_identity_association = true

  # Creates the node IAM role + instance profile and the SQS interruption queue
  node_iam_role_additional_policies = {
    AmazonSSMManagedInstanceCore = "arn:aws:iam::aws:policy/AmazonSSMManagedInstanceCore"
  }
}

# Then `helm_release` installs the chart and points it at:
#   settings.clusterName        = module.eks.cluster_name
#   settings.interruptionQueue  = module.karpenter.queue_name
#   serviceAccount role/identity = module.karpenter.iam_role_arn
```

#### eksctl (one-shot scaffolding)

```bash
# eksctl can create the OIDC provider, controller IAM role, node role,
# SQS queue, and EventBridge interruption rules in one step:
eksctl create iamserviceaccount --cluster my-cluster ...   # or:
eksctl utils associate-iam-oidc-provider --cluster my-cluster --approve
# See the Karpenter "Getting Started with eksctl" guide for the full template.
```

#### Helm values worth setting

```yaml
# values.yaml passed to the karpenter chart
settings:
  clusterName: my-cluster
  interruptionQueue: my-cluster        # SQS queue name
replicas: 2                            # HA: leader-elected controller
controller:
  resources:
    requests: { cpu: "1", memory: 1Gi }
    limits:   { memory: 1Gi }
# Pin Karpenter to a stable managed node group so it never schedules on its own nodes:
nodeSelector: { karpenter.sh/controller: "true" }
tolerations: [{ key: CriticalAddonsOnly, operator: Exists }]
```

> Managed alternative: if you don't want to run the controller, **EKS Auto Mode** (GA 2024) embeds a Karpenter-based, AWS-managed data plane — same NodePool/NodeClass concepts, no controller to operate.

## Monitoring and Observability

### Monitoring Karpenter in Production

#### Key Metrics to Track

Karpenter exposes Prometheus metrics on `:8080/metrics` (scrape with a `ServiceMonitor`):

- **`karpenter_nodepools_usage` / `karpenter_nodepools_limit`**: how close each NodePool is to its `limits` ceiling — alarm before you cap out.
- **`karpenter_nodeclaims_created_total` / `_terminated_total`**: provisioning and disruption volume; a spike in churn often means flapping or bad budgets.
- **`karpenter_voluntary_disruption_decisions_total`**: consolidation/drift/expiration activity by reason.
- **`karpenter_pods_state` / scheduling-simulation duration**: pods stuck Pending and how long the solver takes.
- **`karpenter_cloudprovider_errors_total`**: EC2 API errors (e.g. insufficient-capacity, throttling) that block launches.
- **`karpenter_interruption_*`**: Spot/maintenance interruption handling throughput.

#### Logging Best Practices

- **Watch the controller logs** (`-l app.kubernetes.io/name=karpenter`) — they explain *why* a node was/ wasn't launched (constraints unmet, no capacity, discovery errors) and every disruption decision.
- **Set `LOG_LEVEL=debug`** temporarily when debugging scheduling; revert in steady state to control volume.
- **Alarm on persistently Pending pods** and on `cloudprovider_errors_total` — these surface capacity/IAM/subnet problems early.
- **Dashboard cost vs. NodePool usage** — pair Karpenter metrics with Cost Explorer/Kubecost to confirm consolidation is actually saving money.

## Troubleshooting

### Common Issues with Karpenter

#### Issue 1: Pods stay Pending and no node is created

**Symptoms**: Workload pods remain `Pending`; `kubectl get nodeclaims` shows nothing new.

**Cause**: No instance type satisfies the combined pod requirements/NodePool constraints, the NodePool hit its `limits`, or subnet/SG selectors match nothing.

**Solution**: Read the Karpenter logs and `kubectl describe pod` events. Broaden NodePool requirements, raise `limits`, and verify `kubectl get ec2nodeclass default -o yaml` shows discovered subnets/security groups (check the `karpenter.sh/discovery` tags).

#### Issue 2: Nodes churn / flap constantly

**Symptoms**: Nodes are created and torn down repeatedly; workloads see frequent rescheduling.

**Cause**: Aggressive `consolidateAfter`, missing pod requests causing repacking thrash, or Drift firing on a moving AMI alias.

**Solution**: Lengthen `consolidateAfter`, add/cap disruption **budgets**, set proper requests, and pin the AMI (`amiSelectorTerms` with a fixed id/version) instead of `@latest` if drift is unwanted.

#### Issue 3: Spot interruptions cause hard disruptions

**Symptoms**: Pods are killed abruptly when Spot capacity is reclaimed; no graceful drain.

**Cause**: The interruption SQS queue / EventBridge rules aren't configured, so Karpenter never receives the 2-minute warning.

**Solution**: Create the queue and rules (the Terraform/eksctl modules do this) and set `settings.interruptionQueue`. Confirm `karpenter_interruption_*` metrics increment and that nodes get cordoned ahead of reclaim.

## Comparison with Alternatives

### How Karpenter Compares to Other Solutions

| Dimension | Karpenter | Cluster Autoscaler (CAS) | EKS Managed Node Groups | EKS Auto Mode |
|-----------|-----------|--------------------------|-------------------------|---------------|
| Model | Groupless, just-in-time per-pod provisioning | Scales fixed ASGs/node groups | Static/manually-scaled node groups | Managed Karpenter (AWS-operated) |
| Instance selection | Dynamic, cheapest-that-fits across many types | You pre-define types per group | You pick the type | Dynamic, like Karpenter |
| Scale-up speed | Fast (direct EC2 Fleet, ~seconds–minutes) | Slower (ASG desired-count round-trip) | Manual / slow | Fast |
| Consolidation | Built-in, continuous | Limited scale-down only | None | Built-in |
| Spot handling | Native, price-capacity-optimized + SQS drain | Basic, per-ASG | Manual | Native |
| Ops overhead | One controller + NodePools | Tune many node groups | Per-group tuning | Lowest (no controller to run) |
| Portability | AWS-mature; community providers elsewhere | Multi-cloud, mature everywhere | EKS only | EKS only |

### When to Choose This Tool

Choose **Karpenter** when:

- You're on **EKS** and want fast, right-sized, cost-optimized autoscaling without maintaining a zoo of node groups.
- Your workloads are **heterogeneous or bursty** (ML/GPU, batch, CI) and benefit from dynamic instance selection and consolidation.
- You want **deep Spot savings** with graceful interruption handling.

Prefer the **Cluster Autoscaler** for multi-cloud portability or when you must scale specific pre-defined node groups; prefer **EKS Auto Mode** when you'd rather AWS run the Karpenter control plane for you; and plain **managed node groups** for small, static clusters where any autoscaler is unnecessary.

## Resources

### Official Documentation

- Karpenter docs (concepts, NodePools, disruption): https://karpenter.sh/docs/
- Getting Started with Karpenter on EKS: https://karpenter.sh/docs/getting-started/getting-started-with-karpenter/
- NodePool API reference: https://karpenter.sh/docs/concepts/nodepools/
- EC2NodeClass API reference: https://karpenter.sh/docs/concepts/nodeclasses/
- Disruption (consolidation, drift, budgets): https://karpenter.sh/docs/concepts/disruption/

### Tutorials and Guides

- v1 migration guide (v1beta1 → v1): https://karpenter.sh/docs/upgrading/v1-migration/
- AWS blog — Karpenter graduates to v1.0: https://aws.amazon.com/blogs/containers/karpenter-graduates-to-v1-0/
- EKS Best Practices — Karpenter: https://docs.aws.amazon.com/eks/latest/best-practices/karpenter.html
- terraform-aws-eks Karpenter submodule: https://github.com/terraform-aws-modules/terraform-aws-eks/tree/master/modules/karpenter

### Community Resources

- GitHub (kubernetes-sigs/karpenter + aws/karpenter-provider-aws): https://github.com/aws/karpenter-provider-aws
- Kubernetes Slack `#karpenter`: https://kubernetes.slack.com/
- AWS Containers blog: https://aws.amazon.com/blogs/containers/

### Related Technologies

- Cluster Autoscaler — the predecessor, ASG/node-group based: https://github.com/kubernetes/autoscaler
- EKS Auto Mode — AWS-managed Karpenter data plane
- Kubernetes scheduler / pod requests — what Karpenter provisions against